In [1]:
import pandas as pd
import os
import numpy as np

# ==========================================
# 1. Configuration and Constants
# ==========================================

# Behavior Classification Thresholds (Liters/person/day)
THRESHOLD_ENVIRONMENTALIST = 100.0
THRESHOLD_WASTEFUL = 121.5

# Base Paths (Uncomment the active path)
BASE_PATH = '..' # Currently active path

def clean_income_value(value):
    """
    Converts income string values to float, handling 'X' (anonymized data) as NaN.
    """
    if isinstance(value, str):
        if value.strip().upper() == 'X':
            return np.nan
        # Standardize decimal separator
        value = value.replace('.', '').replace(',', '.')
    try:
        return float(value)
    except (ValueError, TypeError):
        return np.nan

def create_sector_mapping(df, sector_column='CD_SETOR_ORIGINAL', output_dir='..', mapping_filename='de_para_setores_consumidores.csv'):
    """
    Creates a unique sector mapping to S000 format (S001, S002, ...)
    and saves a CSV file with the DE-PARA mapping for future reference.
    Includes household count per sector.
    """
    
    # Get unique sectors and sort for consistency
    unique_sectors = sorted(df[sector_column].dropna().unique())
    
    # Calculate number of households per original sector
    households_per_sector = df.groupby(sector_column).size().to_dict()
    
    # Create mapping: original -> S001, S002, ...
    sector_mapping = {}
    for i, sector in enumerate(unique_sectors, start=1):
        new_id = f'S{i:03d}'  # Always starts at S001
        sector_mapping[sector] = new_id
    
    # Apply mapping to DataFrame
    df['CD_SETOR_SIMPLIFICADO'] = df[sector_column].map(sector_mapping)
    
    # Create and save DE-PARA file
    de_para_df = pd.DataFrame([
        {
            'SETOR_ORIGINAL': orig, 
            'SETOR_SIMPLIFICADO': simp,
            'NUM_HOUSEHOLDS': households_per_sector.get(orig, 0)
        }
        for orig, simp in sector_mapping.items()
    ])
    de_para_df = de_para_df.sort_values('SETOR_SIMPLIFICADO').reset_index(drop=True)
    de_para_df['INDEX'] = de_para_df.index + 1
    de_para_df['CREATION_DATE'] = pd.Timestamp.now().strftime('%Y-%m-%d')
    
    # Reorder columns for better readability
    de_para_df = de_para_df[['INDEX', 'SETOR_SIMPLIFICADO', 'SETOR_ORIGINAL', 'NUM_HOUSEHOLDS', 'CREATION_DATE']]
    
    # Save CSV
    mapping_path = os.path.join(output_dir, mapping_filename)
    de_para_df.to_csv(mapping_path, index=False, sep=';', encoding='utf-8-sig')
    
    print(f"\n--- Sector Mapping ---")
    print(f"Unique sectors mapped: {len(unique_sectors)}")
    print(f"Format: S001 to S{len(unique_sectors):03d}")
    print(f"DE-PARA file saved at: {mapping_path}")
    print(f"Total households mapped: {de_para_df['NUM_HOUSEHOLDS'].sum()}")
    
    return df, sector_mapping


def main():
    print("--- Starting Advanced Behavior Classification (with Census Data) ---")

    # ==========================================
    # 2. Load and Process Consumer Data
    # ==========================================
    consumers_path = os.path.join(BASE_PATH, 'includes', 'Tabela_consumidores_Itapua_com_setor.csv')
    consumers_df = pd.read_csv(consumers_path, sep=',')
    
    # ==========================================
    # 3. Load and Process Income Data
    # ==========================================
    income_path = os.path.join(BASE_PATH, 'includes','ibge_censo2022', 'Agregados_por_setores_renda_responsavel_BR.csv')
    income_df = pd.read_csv(income_path, sep=';')
    
    # Prepare Income Data
    income_df['CD_SETOR'] = income_df['CD_SETOR'].astype(str)
    income_df = income_df.rename(columns={'V06004': 'VL_RENDA_MEDIA_RESPONSAVEL'})
    income_df['VL_RENDA_MEDIA_RESPONSAVEL'] = income_df['VL_RENDA_MEDIA_RESPONSAVEL'].apply(clean_income_value)
    income_df = income_df[['CD_SETOR', 'VL_RENDA_MEDIA_RESPONSAVEL']]

    # ==========================================
    # 4. Load and Process Consumption History
    # ==========================================
    consumption_path = os.path.join(BASE_PATH, 'includes','dados', 'Tabela_consumo_Itapua_120m.csv')
    consumption_df = pd.read_csv(consumption_path, sep=';')

    # Calculate average consumption of the last 12 months
    consumption_df['AM_REFERENCIA'] = pd.to_datetime(consumption_df['AM_REFERENCIA'], format='%Y%m')
    consumption_df = consumption_df.sort_values(by=['SK_MATRICULA', 'AM_REFERENCIA'], ascending=[True, False])
    
    last_12_months = consumption_df.groupby('SK_MATRICULA').head(12)
    avg_consumption = last_12_months.groupby('SK_MATRICULA')['HCLQTCON'].mean().reset_index()
    avg_consumption.rename(columns={'HCLQTCON': 'NN_MEDIA_CONSUMO'}, inplace=True)

    # Merge Consumption Average
    consumers_df = pd.merge(consumers_df, avg_consumption, on='SK_MATRICULA', how='left')

    # ==========================================
    # 5. Merge Income Data
    # ==========================================
    consumers_df['CD_SETOR_ORIGINAL'] = consumers_df['CD_SETOR'].astype(str)
    consumers_df = pd.merge(consumers_df, income_df, left_on='CD_SETOR_ORIGINAL', right_on='CD_SETOR', how='left', suffixes=('_x', '_income'))
    
    consumers_df = consumers_df.drop(columns=['CD_SETOR_income'])
    consumers_df = consumers_df.rename(columns={'CD_SETOR_x': 'CD_SETOR'})

    # ==========================================
    # 6. Load and Process Census Data (IBGE)
    # ==========================================
    census_path = os.path.join(BASE_PATH, 'includes', 'ibge_censo2022', 'Agregados_por_setores_caracteristicas_domicilio1_BR.csv')
    
    # Robust loading (handling separator variations)
    try:
        census_df = pd.read_csv(census_path, sep=';', dtype={'CD_setor': str})
    except pd.errors.ParserError:
        census_df = pd.read_csv(census_path, sep=',', dtype={'CD_setor': str})

    # Rename IBGE columns
    census_df = census_df.rename(columns={'V00005': 'NN_MEDIA_MORADORES_IBGE', 'V00001': 'NN_MEDIA_DOMICILIOS_IBGE'})
    census_df = census_df[['CD_setor', 'NN_MEDIA_MORADORES_IBGE', 'NN_MEDIA_DOMICILIOS_IBGE']]
    
    census_df['CD_SETOR'] = census_df['CD_setor'].astype(str)
    census_df['NN_MEDIA_MORADORES_IBGE'] = pd.to_numeric(census_df['NN_MEDIA_MORADORES_IBGE'], errors='coerce')
    census_df['NN_MEDIA_DOMICILIOS_IBGE'] = pd.to_numeric(census_df['NN_MEDIA_DOMICILIOS_IBGE'], errors='coerce')

    # Calculate Average Residents per Household in the Sector
    census_df['NN_MEDIA_MORADORES_IBGE'] = round(census_df['NN_MEDIA_MORADORES_IBGE'] / census_df['NN_MEDIA_DOMICILIOS_IBGE'], 1)

    # Merge Census Data
    consumers_df = pd.merge(consumers_df, census_df, left_on='CD_SETOR_ORIGINAL', right_on='CD_SETOR', how='left', suffixes=('_app', '_ibge'))
    
    consumers_df = consumers_df.drop(columns=['CD_SETOR_ibge'])
    consumers_df = consumers_df.rename(columns={'CD_SETOR_app': 'CD_SETOR'})

    # ==========================================
    # 7. Data Cleaning and Imputation
    # ==========================================
    
    # Remove records with missing Sector ID
    consumers_df = consumers_df.dropna(subset=['CD_SETOR'])
    consumers_df = consumers_df[consumers_df['CD_SETOR'].astype(str) != '']

    # Create Simplified Sector ID (SXXX)
    consumers_df, sector_mapping = create_sector_mapping(
        df=consumers_df,
        sector_column='CD_SETOR_ORIGINAL',
        output_dir=BASE_PATH + '\\resultados',
        mapping_filename='de_para_setores_consumidores.csv'
    )
    
    # --- IMPUTATION LOGIC FOR RESIDENTS ---
    # Create analysis column
    consumers_df['NN_MORADORES_ANALISE'] = consumers_df['NN_MORADORES'].copy()
    
    # Condition: If registered residents == 0, replace with IBGE Sector Average
    condition_replace = (consumers_df['NN_MORADORES'] == 0) & (consumers_df['NN_MEDIA_MORADORES_IBGE'].notna())
    
    consumers_df.loc[condition_replace, 'NN_MORADORES_ANALISE'] = \
        consumers_df.loc[condition_replace, 'NN_MEDIA_MORADORES_IBGE'].round(0)
        
    # If still NaN or 0, fallback to 1 resident
    consumers_df['NN_MORADORES_ANALISE'] = consumers_df['NN_MORADORES_ANALISE'].fillna(1)
    consumers_df.loc[consumers_df['NN_MORADORES_ANALISE'] == 0, 'NN_MORADORES_ANALISE'] = 1
    consumers_df['NN_MORADORES_ANALISE'] = consumers_df['NN_MORADORES_ANALISE'].astype(int)

    # ==========================================
    # 8. Behavior Classification
    # ==========================================
    
    # Calculate Daily Consumption (Liters per person per day)
    # Formula: (Avg m3 * 1000) / Residents / 30.5 days
    consumers_df['NN_CONSUMO_DIARIO'] = (
        (consumers_df['NN_MEDIA_CONSUMO'] * 1000 / consumers_df['NN_MORADORES_ANALISE']) / 30.5
    )
    
    # Classify Profiles
    consumers_df['TP_COMPORTAMENTO'] = consumers_df['NN_CONSUMO_DIARIO'].apply(
        lambda val: 
            'AMBIENTALISTA' if val < THRESHOLD_ENVIRONMENTALIST
            else 'MODERADO' if THRESHOLD_ENVIRONMENTALIST <= val <= THRESHOLD_WASTEFUL 
            else 'PERDULARIO'
    )

    # ==========================================
    # 9. Final Organization and Main Export
    # ==========================================
    
    columns_order = [
        'SK_MATRICULA', 'NM_LOCALIDADE', 'NM_CATEGORIATARIFARIA', 
        'NM_SITUACAO_IMOVEL', 'NN_MORADORES', 'NN_MORADORES_ANALISE', 
        'ST_PISCINA', 'LAT_GEO', 'LONG_GEO', 'X', 'Y', 
        'CD_SETOR_ORIGINAL', 'CD_SETOR_SIMPLIFICADO', 
        'NN_MEDIA_CONSUMO', 'NN_CONSUMO_DIARIO', 'TP_COMPORTAMENTO', 
        'VL_RENDA_MEDIA_RESPONSAVEL', 'NN_MEDIA_MORADORES_IBGE'
    ]
    
    # Ensure columns exist
    for col in columns_order:
        if col not in consumers_df.columns:
            consumers_df[col] = None
    
    consumers_df = consumers_df[columns_order]
    consumers_df = consumers_df.rename(columns={'CD_SETOR_SIMPLIFICADO': 'CD_SETOR'})

    # Save Main File
    main_output_path = os.path.join(BASE_PATH, 'includes', 'Tabela_consumidores_Itapua_com_setor_comportamento_e_renda.csv')
    consumers_df.to_csv(main_output_path, sep=';', index=False)
    print(f"Main dataset saved: {main_output_path}")

    

    print("\nProcessing Complete.")

if __name__ == "__main__":
    main()

--- Starting Advanced Behavior Classification (with Census Data) ---


C:\Users\Edmilson\AppData\Local\Temp\ipykernel_11780\904663870.py:135: DtypeWarning: Columns (1,3,4,5,7,8,9,10,11,13,14,15,16,17,20,21,28,29,30,31,32,33,34,35,36,37,38,39,40,41,42,43,44,45,46,47,51,52,53,54,56,57,59,60,61,62,63,64,65,66,67,68,69,70,71,72,73,74,77,78,84,87) have mixed types. Specify dtype option on import or set low_memory=False.
  census_df = pd.read_csv(census_path, sep=';', dtype={'CD_setor': str})



--- Sector Mapping ---
Unique sectors mapped: 103
Format: S001 to S103
DE-PARA file saved at: ..\resultados\de_para_setores_consumidores.csv
Total households mapped: 19630


Main dataset saved: ..\includes\Tabela_consumidores_Itapua_com_setor_comportamento_e_renda.csv

Processing Complete.
